<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>


# Robot Operating System 2 Graph and Software Layers

The software running on robots performs many tasks at the same time. For example, Duckiedrones collect sensor readings, process that data into information, use it to estimate their state in the world, and control their motion though motors concurrently. While it is possible to combine all of that work in one monolothic program, separating it into focused components that can easily exchange data makes each part easier to inspect, test, and change down the line.

Robot Operating System 2 ([ROS 2](https://www.ros.org/)) provides libraries and tools that help these components exchange data through agreed interfaces. ROS is a robotics middleware (sitting between hardware and software), sometimes referred to as to as "the McDonalds of robotics" because it widely known and used, and despite it not providing the best performance amongst middlewares, it offers consistent outcomes across the board. It is also a great tool for learning robotics, as it provides a lot of functionality out of the box.

In this LX we will explore ROS 2 through a simplified camera-to-controller example, to understand how the software components cooperate and which responsibilities belong to the application, which to ROS 2, and which to the operating system.

This first notebook is a conceptual introduction. You do not need to run ROS 2 commands yet.

## The ROS 2 graph

A __node__ is a named software component participating in a ROS 2 application. Giving each node a focused responsibility makes its behavior easier to inspect, test, and change. In other words, a node is a piece of software with well defined inputs and outputs, a clear purpose, and a name that identifies it in the ROS 2 graph. Nodes can be written in different programming languages, such as Python or C++.

Suppose we are tasked to take images from a camera, do some processing to estimate the Duckiedrone's position and orientation, and then use that information to control its motion. We could separate those responsibilities into three nodes: a camera node, a pose-estimation node, and a controller node.

The purpose of the camera node is to produce images. A pose-estimation node uses those images to estimate the Duckiedrone's position and orientation, together called its __pose__. A controller uses that estimate and some other information (e.g., a setpoint) to calculate motion commands.

Nodes exchange data in __messages__. A message has a defined __type__, which specifies its fields. For a stream of data, a node __publishes__ messages on a named __topic__, and interested nodes __subscribe__ to that topic.

In other words:
- A "topic" in ROS is like a pipe where data flows. It has a name and a type, and nodes can publish or subscribe to it.
- A "message" is the actual data being sent through that pipe. It has a specific structure defined by its type, which tells the system what kind of data to expect.
- "Publishing" means sending data into the topic, while "subscribing" means listening for data coming out of the topic. Note that multiple nodes can publish to the same topic, and multiple nodes can subscribe to the same topic. This allows for flexible communication patterns in a ROS 2 system.

These connections form the ROS 2 graph. The simplified data flow appears in [Figure 1](#figure-1).

<a id="figure-1"></a>

```text
camera node
    |
    | publishes image messages
    v
image topic
    |
    | subscribed to by
    v
pose-estimation node
    |
    | publishes pose messages
    v
pose topic
    |
    | subscribed to by
    v
controller node
```

_Figure 1: A simplified ROS 2 graph connecting a camera, pose estimator, and controller._

Additional ROS2 features such as __services__ model a request and response. __Actions__ represent longer-running work with a goal, feedback, a result, and cancellation. __Parameters__ hold configuration values for a node.

The pose-estimation node is both a subscriber and a publisher: it receives images and produces pose estimates. The controller consumes those estimates without needing to know how the image-processing algorithm computed them.

This is what makes the separation useful. We can change the estimation algorithm while preserving the interface the controller expects. That agreement includes the topic, message type, and meaning of the data, such as the units and coordinate frame used for the estimate.

### Trace the dependency

Suppose the camera continues publishing images, but the pose-estimation node stops producing messages. Which input would the controller lose? Would the presence of image messages alone establish that the whole data path is working?

<details>
<summary>Compare your reasoning</summary>

The controller would lose new pose estimates. Image messages show that data is available at the beginning of this path, but we would also need to inspect the estimator's output to establish whether data reaches the controller.

</details>

## Software layers

The nodes contain the application logic: reading a camera, estimating a pose, or calculating a command. ROS 2 supplies common communication facilities so each application does not have to implement them from scratch. 

A Python node uses the ROS 2 client library, `rclpy`, to create publishers and subscriptions. Beneath the client library, the ROS Middleware (RMW) interface connects ROS 2 to a middleware implementation. That implementation handles communication and discovery, the process through which participants find one another.

The relationship between these layers appears in [Figure 2](#figure-2).

<a id="figure-2"></a>

```text
application logic (i.e., what is the software actual meant to do?)
    |
Python client library: rclpy 
    |
common ROS client-library layers
    |
ROS Middleware (RMW) interface
    |
middleware implementation
```

_Figure 2: A simplified path from Python application code to the communication middleware._

Despite its name, ROS 2 does not replace the operating system. In this LX, it runs on Linux, which provides facilities such as process scheduling, device access, and networking. ROS 2 builds on those facilities to connect the application nodes.

For example, your publisher code chooses the message contents and when to publish. It calls `rclpy` (if using Python, or `rclcpp` for C++)to publish the message; the communication layers handle its exchange with compatible subscribers.

## ROS 2 is distributed

The graph describes **communication relationships**, rather than where programs run. Several nodes can share one process, run in separate processes on one computer, or communicate across computers or with "the cloud".

Moving the pose estimator to another computer would possibly change some performance metrics such as latency or data publication frequency, but not its role in our example: it would still subscribe to images and publish poses. Communication would also require compatible discovery settings and a working network connection.

Later notebooks will use the `ros2` command-line tools to inspect the graph visible from a terminal and check whether messages actually arrive.

## Further reading

The ROS 2 Jazzy documentation introduces [ROS 2 concepts](https://docs.ros.org/en/jazzy/Concepts.html), explains [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html), and describes the available [middleware implementations](https://docs.ros.org/en/jazzy/Concepts/Advanced/About-Middleware-Implementations.html).

For additional insight on the difference between logical and physical architectures, you can check this Duckietown lecture on [System Architectures](https://hubs.ly/Q02b8kCV0). 

## Checkpoint

Run the self-check in the next cell. Write a response before revealing the answer.

<!--
**1. You replace the pose-estimation algorithm. What must remain compatible for the existing controller to keep using its output, and why?**

**Model answer:** The controller still needs the expected topic, message type, and meaning of the pose data, including units and coordinate frame. Keeping that interface compatible allows the controller to consume the estimates without depending on the estimator's internal algorithm.

**2. In a Python publisher running on Linux, which responsibilities belong to the application code, ROS 2, and the operating system?**

**Model answer:** The application decides what data to publish and when. ROS 2 provides the node and publisher interfaces and the underlying discovery and message-exchange facilities. Linux supplies the process, device, and networking facilities on which the software runs.
-->
## Summary and next steps

A ROS 2 graph connects focused nodes through shared interfaces. Following its messages helps you identify which component produces the data another component needs. The client library and middleware support those exchanges, while Linux provides the underlying operating-system facilities.

Our example started with a camera node already publishing ROS 2 messages. On a Duckiedrone, sensor data can pass through other communication systems before reaching the ROS 2 graph. [Notebook 2: ROS 2 Data Paths and Duckiedrone Integration](./2-ros-2-data-paths-and-duckiedrone-integration.ipynb) follows those routes.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
